# 02 — Tre approcci operativi (tutti liberi/aperti)
1. **Modello open locale** — Ollama / llama.cpp con modelli come Laya, Kev o Tev1.
2. **Route ospitata gratuita** — un endpoint con piano free, solo se serve.
3. **Lettura dei logit senza training** — metodi open (LitJev, SemIf, AnyJev) leggono le probabilità delle opzioni da qualunque modello open senza fine-tuning.

Il mock offline fa da sostituto così il notebook gira ovunque.


In [ ]:
from decision_qa import load_config, SystemOneClient, MockClient

local_cfg = load_config('configs/local_ollama.yaml')
hosted_cfg = load_config('configs/hosted.yaml')
print('local :', local_cfg.backend, local_cfg.base_url, local_cfg.model)
print('hosted:', hosted_cfg.backend, hosted_cfg.base_url, hosted_cfg.model)


## Confronto sugli stessi input
Misuriamo il tempo di ogni client sulle stesse domande. Una chiamata reale avviene solo se un server locale è attivo; altrimenti si usa il mock.


In [ ]:
import time
from decision_qa import NoulQuestion, ChoiceQuestion

state = {'context': 'The context says the treatment helped.', 'question': 'Did it help?'}
qs = [
    NoulQuestion(label='helped', statement='The treatment helped.', gold=True),
    ChoiceQuestion(label='verdict', options=['yes', 'no', 'unclear'], gold_index=0),
]

def timed(client, label):
    t0 = time.perf_counter()
    results = client.decision(state, qs)
    dt = (time.perf_counter() - t0) * 1000
    print(f'{label:12s} {dt:7.1f} ms', {r.question_label: round(max(r.probabilities.values()), 3) for r in results})

timed(MockClient(seed=0), 'mock')
try:
    timed(SystemOneClient(local_cfg), 'local')
except Exception as e:
    print('local endpoint not running:', type(e).__name__, '- start Ollama to try it')


## Compromessi
| Approccio | Costo | Setup | Ideale per |
|---|---|---|---|
| Modello open locale | gratuito | scaricare i pesi | privacy, offline |
| Route ospitata free | piano gratuito | account free | senza hardware locale |
| Lettura logit | gratuito | modello open | prototipi rapidi |

Implementazioni open: [LitJev](https://github.com/zhengxuyu/litjev), [SemIf](https://github.com/TheoLeeCJ/SemIf-OpenJev), [AnyJev](https://github.com/nokia-applied-research/AnyJev).
